# Is it a detection? Frequentist hypothesis testing

Last week we saw that the SNR is a random variable: the same signal in two different noise realisations gives two different SNRs. This week we use that fact to answer the question every search has to answer: **when is a loud SNR a real detection, and how loud must a signal be to be detected?**

This notebook follows Section 5.2 of S. R. Taylor, *The Nanohertz Gravitational Wave Astronomer* ([arXiv:2105.13270](https://arxiv.org/abs/2105.13270)). In the exercise you compute the probabilities with **Monte Carlo integration**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import i0e

%config InlineBackend.figure_format = 'retina'

## 1. The detection statistic and its two distributions

In frequentist statistics the experiment is imagined to be **repeatable**: the probability of an outcome is the fraction of identical trials in which it happens. From the data $d$ we compute a **detection statistic** $S(d)$, a single number that says how much the data look like a signal. For GW searches the natural choice is the matched-filter SNR, $S = \rho$.

Because the noise is random, $\rho$ changes from trial to trial. What matters is its **sampling distribution** under two hypotheses:
- $\mathcal{H}_\mathrm{null}$: the data are only noise, $d = n$;
- $\mathcal{H}_\mathrm{signal}$: the data contain a signal, $d = n + h$.

These distributions can be computed analytically or by simulating many noise realisations (as you did last week). In this notebook we use a simple **toy model**. Take it as given:

$$p(\rho\,|\,\mathcal{H}_\mathrm{null}) = \rho\, e^{-\rho^2/2}, \qquad\qquad p(\rho\,|\,A;\,\mathcal{H}_\mathrm{signal}) = \rho\, e^{-(\rho^2 + A^2)/2}\, I_0(A\rho), \qquad \rho \ge 0,$$

where $I_0$ is the modified Bessel function of the first kind and $A \ge 0$ is the **amplitude** of the signal (how loud it is: roughly the SNR you would get without noise). For $A = 0$ the signal distribution reduces to the null distribution, since $I_0(0) = 1$.

*(Aside: these are the Rayleigh and Rice distributions. The SNR maximised over the unknown phase of the signal follows exactly these, but nothing below depends on that.)*

In [ ]:
rho = np.linspace(0, 10, 1000)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rho, stats.rayleigh.pdf(rho), color="k", lw=2, label=r"$\mathcal{H}_\mathrm{null}$")
for A in [2, 4, 6]:
    ax.plot(rho, stats.rice.pdf(rho, A), lw=2, label=rf"$\mathcal{{H}}_\mathrm{{signal}}$, $A = {A}$")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel(r"$p(\rho\,|\,\mathcal{H})$")
ax.set_xlim(0, 10)
ax.set_ylim(0)
ax.legend(frameon=False)
plt.show()

The louder the signal, the further its distribution moves to the right. The width stays about 1: noise always scatters the SNR by about $\pm 1$. Notice that the distributions **overlap**. A single value of $\rho$ cannot tell us for sure which hypothesis is true, only how probable it is under each.

## 2. Significance: the $p$-value

We measure $\rho_\mathrm{obs}$. How often could noise alone give a value at least this large? This is the **$p$-value**:

$$p = P(\rho > \rho_\mathrm{obs}\,|\,\mathcal{H}_\mathrm{null}) = \int_{\rho_\mathrm{obs}}^{\infty} p(\rho\,|\,\mathcal{H}_\mathrm{null})\, d\rho . \tag{5.2}$$

It is the area of the tail of the null distribution beyond the observed value.

In [ ]:
rho_obs = 2.5
tail = rho >= rho_obs

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rho, stats.rayleigh.pdf(rho), color="k", lw=2, label=r"$p(\rho\,|\,\mathcal{H}_\mathrm{null})$")
ax.fill_between(rho[tail], stats.rayleigh.pdf(rho[tail]), color="0.6", label=r"$p = P(\rho > \rho_\mathrm{obs}\,|\,\mathcal{H}_\mathrm{null})$")
ax.axvline(rho_obs, color="crimson", ls="--", label=r"$\rho_\mathrm{obs}$")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel("pdf")
ax.set_xlim(0, 6)
ax.set_ylim(0)
ax.legend(frameon=False)
plt.show()

A small $p$-value means that noise rarely produces such a loud $\rho$. Be careful: a small $p$-value argues **against the null hypothesis**. It is *not* the probability that the signal is real, and it says nothing about the signal hypothesis. For that we need $p(\rho\,|\,\mathcal{H}_\mathrm{signal})$ too.

## 3. Type I and type II errors

To decide, we fix a **threshold** $\rho_\mathrm{th}$ before looking at the data, and claim a detection whenever $\rho_\mathrm{obs} > \rho_\mathrm{th}$. Two mistakes are possible:
- **Type I error (false alarm):** noise alone crosses the threshold, $\rho_\mathrm{obs} > \rho_\mathrm{th}$ under $\mathcal{H}_\mathrm{null}$;
- **Type II error (false dismissal):** a real signal stays below the threshold, $\rho_\mathrm{obs} < \rho_\mathrm{th}$ under $\mathcal{H}_\mathrm{signal}$.

Their probabilities are

$$\alpha = P(\rho > \rho_\mathrm{th}\,|\,\mathcal{H}_\mathrm{null}), \qquad \beta = P(\rho < \rho_\mathrm{th}\,|\,\mathcal{H}_\mathrm{signal}). \tag{5.3}$$

The **detection probability** is $\mathrm{DP} = 1 - \beta$: the fraction of trials in which a signal that is really there is detected.

In [ ]:
rho_th, A = 3.0, 4.0
above, below = rho >= rho_th, rho <= rho_th
p0, p1 = stats.rayleigh.pdf(rho), stats.rice.pdf(rho, A)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rho, p0, color="k", lw=2, label=r"$p(\rho\,|\,\mathcal{H}_\mathrm{null})$")
ax.plot(rho, p1, color="C0", lw=2, label=rf"$p(\rho\,|\,A={A:g};\,\mathcal{{H}}_\mathrm{{signal}})$")
ax.fill_between(rho[above], p0[above], color="crimson", alpha=0.5, label=r"$\alpha$ (false alarm)")
ax.fill_between(rho[below], p1[below], color="C0", alpha=0.3, label=r"$\beta$ (false dismissal)")
ax.axvline(rho_th, color="k", ls="--", label=r"$\rho_\mathrm{th}$")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel("pdf")
ax.set_xlim(0, 8)
ax.set_ylim(0)
ax.legend(frameon=False)
plt.show()

Moving the threshold to the right lowers $\alpha$ but raises $\beta$: you cannot reduce both at once for a given signal. Only a louder signal (larger $A$) moves its distribution away from the null one and makes $\beta$ small.

A common but weak claim is "the *expected* SNR of the signal is above threshold". If the signal distribution is centred on $\rho_\mathrm{th}$, then $\rho_\mathrm{obs}$ is above threshold in only about half of the trials: a detection probability of about 50%, no better than a coin flip.

## 4. Upper limits

If nothing is detected we can still constrain the amplitude: *if the signal were louder than $A_{95\%}$, we would have measured a larger $\rho$ than $\rho_\mathrm{obs}$ in 95% of the trials*. The frequentist 95% upper limit $A_{95\%}$ is defined by

$$P(\rho \ge \rho_\mathrm{obs}\,|\,A_{95\%};\,\mathcal{H}_\mathrm{signal}) = 0.95 . \tag{5.4}$$

We increase $A$ until 95% of the signal distribution lies above the observed value:

In [ ]:
rho_obs = 2.0
tail = rho >= rho_obs

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, A in zip(axes, [1, 2, 3]):
    p1 = stats.rice.pdf(rho, A)
    ax.plot(rho, p1, color="C0", lw=2)
    ax.fill_between(rho[tail], p1[tail], color="C0", alpha=0.3)
    ax.axvline(rho_obs, color="crimson", ls="--")
    ax.set_title(rf"$A = {A}$:  area $= {stats.rice.sf(rho_obs, A):.2f}$")
    ax.set_xlabel(r"$\rho$")
    ax.set_xlim(0, 7)
    ax.set_ylim(0)
axes[0].set_ylabel(r"$p(\rho\,|\,A;\,\mathcal{H}_\mathrm{signal})$")
plt.show()

None of these reaches 0.95 yet, so $A_{95\%} > 3$ (it is about 3.45).

# False Dismissal Probability

Let us assume that a distribution of SNRs follows the Rice distribution, given by

$$f(x,b) = x\:\text{exp}\big(-\frac{x^2+b^2}{2}\big)I_0(x\:b) $$

Assume the SNR threshold $\rho_{thr} = 3$.

In [ ]:
threshold = 3
rho = np.linspace(0,10,1000)
pdf = stats.rice.pdf(rho, 4)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rho, pdf, lw=2, color="purple", label=r"$f(x,4)$")
mask = rho <= threshold
ax.fill_between(rho[mask], pdf[mask], color="purple", alpha=0.25,label=rf"$P(\rho<\rho_{{\text{{thr}}}})$")
ax.axvline(threshold, color="k", ls="--", label=r"$\rho_{\text{thr}}$")
ax.set_xlabel(r"$\rho$")
ax.set_ylabel("pdf")
ax.set_xlim(0, 8)
ax.set_ylim(0)
ax.legend(frameon=False)
plt.show()

In this exercise we want to demonstrate how you can calculate false dismissal probabilities from your SNR measurements when you don't have access to the functional form of the distribution itself. 

#### **Task 1:** Calculate the false dismissal probability $\beta$ for a SNR threshold $\rho_{thr} = 3$ using N = 100 samples

Start out by using `stats.rice.rvs` to draw N = 100 samples from the Rice distribution. You can look up how to use it here: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rice.html

In [ ]:
N_samples = ...
samples = ...

plt.plot(rho, stats.rice.pdf(rho, 4), lw=2, color = "purple")
plt.hist(samples,bins = 12, density = "normalize", color = "green", alpha = 0.5,label = "samples")
plt.scatter(samples,stats.rice.pdf(samples, 4),color = "green")
plt.axvline(threshold, color="k", ls="--", label=r"$\rho_{\text{thr}}$")
plt.ylabel("probability")
plt.xlabel("SNR")
plt.legend()
plt.show()

Approximate the probability $P(\rho< \rho_{thr})$ as the fraction of samples below that threshold. 

In [ ]:
# TO DO
frac_below = ...
print("estimated integral:", frac_below, "real integral:", stats.rice.cdf(threshold,4))

#### **Task 2:** Plot your estimates as a function of $N$. 

In [ ]:
# TO DO
N_array = ...
estimates = []

for N in N_array:
    ...

In [ ]:
beta_exact = stats.rice.cdf(threshold,4)
plt.plot(N_array,estimates,"o-",color = "purple",label = r"$\beta$ estimates")
plt.axhline(beta_exact, color="crimson", ls="--", label=r"exact $\beta$")
plt.ylabel(r"$\beta$")
plt.xlabel("N")
plt.legend()
plt.show()

## Error of our Estimate

The varaince of a binomial distribution is given by $$\text{Var}(\hat{p}) = \frac{p(1-p)}{N} $$

In [ ]:
beta_exact_array = np.array([beta_exact for i in range(len(N_array))])
error = np.sqrt(beta_exact_array*(1-beta_exact_array)/N_array)
estimate_errors = np.abs(beta_exact_array - estimates)

In [ ]:
plt.plot(N_array, estimate_errors,".",label = "error of estimates")
plt.plot(N_array,error, "-.",color = "grey", label = "theoretical standard deviation")
plt.legend()
plt.xlabel("N")
plt.ylabel("error")
plt.show()